# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset on knowledge adoption in rangeland management, using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install the `mlcroissant` library if not already present
!pip install mlcroissant

## 1. Data Loading
Load the dataset schema and metadata via the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Name: {getattr(metadata, 'name', '')}\nDescription: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. Each entity is uniquely identified via its `@id` for clarity and reproducibility.

In [ ]:
# List all available record sets and their details
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    # List fields in this record set
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field.id} | Name: {field.name} | Data type: {getattr(field, 'data_type', None)}")
    print()

## 3. Data Extraction
Extract record sets of interest into Pandas DataFrames for downstream analysis.

We'll demonstrate loading each record set by their `@id` (see the overview above for a list of available IDs and fields).

In [ ]:
# Gather all record set @id's for extraction
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set @id: {record_set_id} | Loaded records: {len(df)}")

# For demonstration, pick the first available record set for further inspection
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"\nColumns in record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    print(f"\nSample rows:")
    display(dataframes[main_rs_id].head())
else:
    print('No record sets found in this dataset.')

## 4. Exploratory Data Analysis (EDA)
Process and explore parts of the data. You'll often want to filter, normalize, or group records for further analysis.

> Replace the example field IDs below with actual numeric/groupable field `@id`s from the earlier overview (`section 2`) that make sense for your investigation.

In [ ]:
# Example: Filter, normalize, and group on a numeric field

# Use actual field @id's seen in the overview above
# For example purposes we use placeholder IDs:
example_numeric_field_id = None
example_group_field_id = None

if record_set_ids:
    main_df = dataframes[main_rs_id]
    # Attempt to automatically find a numeric field
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            example_numeric_field_id = col
            break
    # Attempt to find a groupable field
    for col in main_df.columns:
        if main_df[col].nunique() < len(main_df)//2 and main_df[col].nunique() > 1 and main_df[col].dtype == object:
            example_group_field_id = col
            if example_group_field_id != example_numeric_field_id:
                break

    if example_numeric_field_id is None:
        print(f"No obvious numeric field found for EDA in record set {main_rs_id}.")
    else:
        threshold = main_df[example_numeric_field_id].mean()
        # Filter records above the mean
        filtered_df = main_df[main_df[example_numeric_field_id] > threshold].copy()
        print(f"Filtered records with {example_numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{example_numeric_field_id}_normalized"] = (
            (filtered_df[example_numeric_field_id] - filtered_df[example_numeric_field_id].mean()) /
            filtered_df[example_numeric_field_id].std()
        )
        print(f"\nNormalized {example_numeric_field_id}:")
        display(filtered_df[[example_numeric_field_id, f"{example_numeric_field_id}_normalized"]].head())

        if example_group_field_id and example_group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(example_group_field_id)[example_numeric_field_id].mean().reset_index()
            print(f"\nMean {example_numeric_field_id} grouped by {example_group_field_id}:")
            display(grouped_df.head())
else:
    print('No record sets loaded for EDA.')

## 5. Visualization
Visualize data distributions and relationships. Replace the placeholder field IDs as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if record_set_ids and example_numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[example_numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {example_numeric_field_id}")
    plt.xlabel(example_numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group if available
    if example_group_field_id and example_group_field_id in main_df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=main_df[example_group_field_id], y=main_df[example_numeric_field_id])
        plt.title(f"{example_numeric_field_id} by {example_group_field_id}")
        plt.xlabel(example_group_field_id)
        plt.ylabel(example_numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Visualization skipped: No numeric field available or no record sets loaded.')

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and process the FAIR² dataset using the `mlcroissant` library. Using the Croissant schema's globally unique `@id`s for all dataset entities ensures reliable, reproducible analysis in data science workflows.